# SC-11-Formal-Verification - Verification Formelle

**Navigation** : [Index](../README.md) | [<< Fuzz Testing](SC-10-Fuzz-Testing.ipynb) | [Move Sui >>](../04-Multi-Chain/SC-12-Move-Sui.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les principes de la **verification formelle**
2. Utiliser **Certora Prover** et le langage CVL
3. Ecrire des **specifications** et des **regles**
4. Verifier des invariants mathematiques

### Duree estimee : 50 minutes

---

## 1. Introduction a la Verification Formelle

La verification formelle prouve mathematiquement la correction d'un programme.

In [ ]:
# Concepts de verification formelle
print("""
VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)
""")

---

## 2. Certora Prover

Certora utilise le CVL (Certora Verification Language).

In [ ]:
# Contrat Solidity a verifier
SOL_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}
'''

print("Contrat Vault a verifier:")
print(SOL_CONTRACT)

In [ ]:
# Specification CVL pour Vault
CVL_SPEC = '''
// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);
    
    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();
    
    // Verifier le resultat
    assert balances(user) == balanceBefore + amount;
}

// Regle: withdraw reduit correctement la balance
rule withdrawDecreasesBalance(address user, uint256 amount) {
    require amount > 0;
    require balances(user) >= amount;
    
    uint256 balanceBefore = balances(user);
    uint256 totalBefore = totalDeposits();
    
    env.msgSender = user;
    withdraw(amount);
    
    assert balances(user) == balanceBefore - amount;
    assert totalDeposits() == totalBefore - amount;
}
'''

print("Specification CVL:")
print(CVL_SPEC)

---

## 3. Commandes Certora

Execution du verifier.

In [ ]:
# Commandes Certora
print("""
INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun.py Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun.py Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun.py Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun.py Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance
""")

---

## 4. Patterns Courants

### 4.1 Conservation des tokens (ERC-20)

In [ ]:
# Specification ERC-20 simplifiee
ERC20_SPEC = '''
methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;n    require balanceOf(from) >= amount;
    
    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();
    
    env.msgSender = from;
    transfer(to, amount);
    
    assert balanceOf(from) == fromBalanceBefore - amount;
    assert balanceOf(to) == toBalanceBefore + amount;
    assert totalSupply() == supplyBefore;
}
'''

print("Specification ERC-20:")
print(ERC20_SPEC)

---

## 5. Exercices

In [ ]:
# Exercice: Verifier un SimpleAuction
EXERCISE_AUCTION = '''
// Contrat
contract SimpleAuction {
    address public beneficiary;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    function bid() external payable {
        require(!ended, "Auction ended");
        require(msg.value > highestBid, "Bid too low");
        highestBidder = msg.sender;
        highestBid = msg.value;
    }

    function end() external {
        require(!ended, "Already ended");
        ended = true;
    }
}

// Specification CVL
/*
invariant highestBidPositive()
    highestBid >= 0;

invariant endedImmutable()
    (ended => always(ended))

rule bidUpdatesHighestBidder(address bidder, uint256 amount) {
    require !ended;
    require amount > highestBid;
    
    env.msgSender = bidder;
    env.msgValue = amount;
    bid();
    
    assert highestBidder == bidder;
    assert highestBid == amount;
}
*/
'''

print("Exercice Auction:")
print(EXERCISE_AUCTION)

---

## 6. Resume

| Concept | Description |
|--------|-------------|
| Invariant | Propriete toujours vraie |
| Regle | Comportement attendu apres une action |
| `methods` | Declaration des fonctions du contrat |
| `envfree` | Fonction sans effets de bord |
| `env` | Environnement (msg.sender, msg.value, |

### Avantages
- Couverture complete (tous les cas possibles)
- Preuves mathematiques rigoureuses
- Detection de bugs subtils

### Limites
- Complexite de configuration
- Temps de verification
- Faux positifs possibles

---

**Notebook suivant** : [SC-12-Move-Sui](../04-Multi-Chain/SC-12-Move-Sui.ipynb)